# Прогноз дохода клиентов банка

**Задача.** По анонимизированным признакам клиента предсказать его официальный доход `target`.

Два независимых механизма складываются: **seed-bag** снижает дисперсию бустинга, **NN-корректор**
добавляет сигнал из категориальных сущностей, недоступный деревьям. Итоговый бленд — их взвешенная сумма.

Решение выполнялось на ноутбуке в visual studio code.

## 0. Импорты и конфигурация

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import time, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb_lib
from sklearn.model_selection import KFold
from sklearn.preprocessing import QuantileTransformer # RankGauss-нормализация входа нейросети
from sklearn.isotonic import IsotonicRegression # реконструкция функции весов w(y)
import torch, torch.nn as nn
torch.set_num_threads(1); torch.set_num_interop_threads(1)
warnings.filterwarnings('ignore')

# устройство нейросети: Colab-GPU (cuda) > Apple MPS > CPU — выбирается автоматически
if torch.cuda.is_available():
    DEV = 'cuda'
elif torch.backends.mps.is_available():
    DEV = 'mps'
else:
    DEV = 'cpu'
SEED = 42
TARGET_MIN, TARGET_MAX = 20_000, 1_500_000 # границы клипа прогноза (диапазон таргета в данных)
TREE_W = np.array([0.464, 0.362, 0.186]) # веса бленда деревьев (LGBM-X1, LGBM-X2, XGB) из OOF
GAMMA = 0.20 # вес NN-корректора в финальном бленде
CAT_FEATURES = ['gender', 'adminarea', 'city_smart_name', 'addrref', 'dp_ewb_last_employment_position']
SERVICE_COLS = ['id', 'dt', 'target', 'w'] # служебные колонки (не признаки)
NOISE_FLOOR = 40.0 # практический шум валидации (порог значимости)
TRAPZ = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

def wmae_np(y_true, y_pred, weights):
    # соревновательная метрика: взвешенная MAE с обязательным клипом прогноза
    return (weights * np.abs(y_true - np.clip(y_pred, TARGET_MIN, TARGET_MAX))).mean()

print(f'torch={torch.__version__}, устройство нейросети = {DEV}')

torch=2.13.0, устройство нейросети = mps


## 1. Загрузка данных

`train.csv` = январь–июнь 2024, `test.csv` = июль–ноябрь 2024.
CSV имеют разделитель `;` и десятичную запятую; часть колонок БКИ хранит числа как строки со смешанным
разделителем — их приводим к числу вручную через `to_numeric`. Пять признаков объявляются
категориальными (LightGBM работает с ними нативно, нейросеть — через эмбеддинги).

In [ ]:
def fix_types(df):
    # приведение типов: строковые числовые колонки -> float; заявленные категории -> category
    df = df.copy()
    for col in df.columns:
        if col in CAT_FEATURES or col in SERVICE_COLS:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
    for col in CAT_FEATURES:
        df[col] = df[col].astype('category')
    return df

def _find(fname):
    for d in ['.', '/content', './final-dataton-alpha']:
        p = os.path.join(d, fname)
        if os.path.exists(p): return p
    raise FileNotFoundError(f'{fname} не найден')

def load_data():
    train_df = pd.read_csv(_find('train.csv'), decimal=',', sep=';', low_memory=False)
    test_df  = pd.read_csv(_find('test.csv'),  decimal=',', sep=';', low_memory=False)
    train_df, test_df = fix_types(train_df), fix_types(test_df)
    # единый словарь категорий на train+test, иначе коды категорий разойдутся между выборками
    for col in CAT_FEATURES:
        cats = pd.api.types.union_categoricals([train_df[col], test_df[col]]).categories
        train_df[col] = train_df[col].cat.set_categories(cats)
        test_df[col]  = test_df[col].cat.set_categories(cats)
    return train_df, test_df

t0 = time.time()
train_df, test_df = load_data()
y_np, w_np = train_df['target'].values, train_df['w'].values # таргет и веса метрики
val_mask = (train_df['dt'] == '2024-06-30').values # июнь = holdout для валидации
print(f'[{time.time()-t0:.0f}s] train {train_df.shape}, test {test_df.shape}')
print(f'corr(w, target) = {np.corrcoef(w_np, y_np)[0,1]:.3f}   (вес почти детерминирован таргетом)')
print(f'строк в июньском holdout = {val_mask.sum()}')

[5s] train (76786, 224), test (73214, 222)
corr(w, target) = 0.716   (вес почти детерминирован таргетом)
строк в июньском holdout = 16214


## 2. Feature engineering (валидированный набор признаков)

Набор фич отобран за многие итерации. Ключевые группы:
* **ранги внутри месяца** (`rankm__*`) — дрейф-устойчивая замена абсолютным значениям доходных прокси;
* **отношения к медианам** по городу / региону / (возраст×пол) — насколько клиент «богаче окружения»;
* **отношения-декодеры дохода** (`incomeValue`/лимит, зарплата/оборот и т.п.);
* **target-encoding региона** (`TE_area`) — fold-safe, чтобы не было утечки таргета;
* **медиана зарплаты по должности** (`pos_sal_med`) — supervised-агрегат по строковой должности;
* **H2-фичи** — «пол дохода» (максимум трат/оборотов) и доход, вменённый из кредитных лимитов.

Итог: `NONSTACK` (131 признак) для деревьев + 6 H2-фич. `M1_BASE` (127 признаков) — подмножество для входа CDF-стека
(из него исключены 4 фичи, которые полезны только финальным деревьям).

In [ ]:
T = 'turn_cur_cr_avg_act_v2'; SAL = 'salary_6to12m_avg'; IV = 'incomeValue'
L = 'hdb_bki_total_max_limit'; P12 = 'dp_ils_paymentssum_avg_12m'; S1Y = 'dp_ils_avg_salary_1y'
GROUP_KEY_FEATS = [T, SAL, IV, L, 'hdb_bki_total_cc_max_limit', 'hdb_bki_total_pil_max_limit',
                   'turn_cur_db_avg_act_v2', P12]
RANKM_FEATS = GROUP_KEY_FEATS + ['dp_payoutincomedata_payout_avg_6_month', 'avg_cur_cr_turn',
                                 S1Y, 'first_salary_income', 'turn_cur_cr_max_v2',
                                 'curr_rur_amt_cm_avg', 'total_rur_amt_cm_avg']
SPEND_CATS = ['avg_by_category__amount__sum__cashflowcategory_name__supermarkety',
              'avg_by_category__amount__sum__cashflowcategory_name__kafe',
              'avg_by_category__amount__sum__cashflowcategory_name__oteli',
              'avg_by_category__amount__sum__cashflowcategory_name__odezhda',
              'avg_by_category__amount__sum__cashflowcategory_name__produkty',
              'avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate']
CASH = SPEND_CATS[-1]

NONSTACK = ["avg_6m_government_services", "turn7d_vs_12m", "incomeValueCategory", "rest_share", "db_range_norm", "transaction_category_supermarket_sum_cnt_m3_4", "dda_rur_amt_3m_avg", "profit_income_out_rur_amt_12m", "turn_save_db_min_v2", "r__incomeValue__hdb_bki_total_cc_max_limit", "avg_by_category__amount__sum__cashflowcategory_name__odezhda", "by_category__amount__sum__eoperation_type_name__perevod_mezhdu_svoimi_schetami", "hdb_relend_outstand_sum", "hdb_bki_active_cc_max_outstand", "bki_total_il_max_limit", "hdb_bki_active_pil_cnt", "min_balance_rur_amt_1m_af", "avg_3m_all", "bki_total_ip_max_outstand", "dp_ils_salary_ratio_1y3y", "avg_by_category__amount__sum__cashflowcategory_name__produkty", "avg_debet_turn_rur", "transaction_category_supermarket_sum_amt_d15", "hdb_outstand_sum", "dp_payoutincomedata_payout_avg_6_month", "dp_payoutincomedata_payout_avg_3_month", "label_Above_1M_share_r1", "hdb_bki_other_active_pil_outstanding", "total_rur_amt_cm_avg_period_days_ago_v2", "transaction_category_restaurants_sum_amt_m2", "r__avg_cur_cr_turn__turn_cur_cr_max_v2", "arearatio__hdb_bki_total_cc_max_limit", "avg_by_category__amount__sum__cashflowcategory_name__set_supermarketov", "transaction_category_supermarket_inc_cnt_2m", "turn_cur_cr_sum_v2", "salary_median_in_gex_r1", "pil", "uniV5", "avg_6m_clothing", "winback_cnt", "hdb_bki_total_products", "cntRegionTripsWavg1m", "turn_save_cr_max_v2", "util_cc", "dp_ils_avg_simultanious_jobs_5y", "turn_cur_db_avg_v2", "hdb_bki_total_auto_max_limit", "turn_cur_db_7avg_avg_v2", "cityratio__hdb_bki_total_max_limit", "turn_other_db_max_v2", "turn_cur_cr_min_v2", "hdb_bki_total_ip_max_limit", "transaction_category_supermarket_percent_cnt_2m", "by_category__amount__sum__eoperation_type_name__vhodjaschij_bystryj_platezh_sbp", "avg_6m_all", "ageratio__hdb_bki_total_pil_max_limit", "blacklist_flag", "avg_by_category__amount__sum__cashflowcategory_name__kafe", "ageratio__hdb_bki_total_cc_max_limit", "r__incomeValue__avg_cur_cr_turn", "dda_rur_amt_curr_v2", "turn_cur_db_min_v2", "adminarea", "avg_by_category__amount__sum__cashflowcategory_name__gipermarkety", "hdb_bki_total_cc_max_limit", "turn_cur_db_avg_act_v2", "dp_ils_avg_salary_3y", "by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona", "avg_cur_db_turn", "mob_cnt_days", "ageratio__incomeValue", "turn_cur_db_sum_v2", "total_rur_amt_cm_avg", "turn_cur_db_max_v2", "city_smart_name", "hdb_bki_active_pil_max_limit", "min_balance_rur_amt_6m_af", "total_sum", "avg_credit_turn_rur", "arearatio__turn_cur_cr_avg_act_v2", "per_capita_income_rur_amt", "avg_cur_cr_turn", "db_stab_avg_max", "cash_share", "hdb_bki_total_ip_cnt", "by_category__amount__sum__eoperation_type_name__ishodjaschij_bystryj_platezh_sbp", "avg_loan_cnt_with_insurance", "cr_stab_avg_max", "rankm__turn_cur_db_avg_act_v2", "rankm__avg_cur_cr_turn", "curr_rur_amt_cm_avg", "label_500k_to_1M_share_r1", "curr_rur_amt_3m_avg", "turn_cur_cr_7avg_avg_v2", "dp_ils_paymentssum_avg_6m_current", "age", "limit_per_product", "rankm__hdb_bki_total_cc_max_limit", "hdb_bki_active_cc_max_limit", "rankm__incomeValue", "ageratio__salary_6to12m_avg", "avg_6m_money_transactions", "cityratio__salary_6to12m_avg", "turn_cur_cr_avg_act_v2", "rankm__hdb_bki_total_max_limit", "turn_cur_cr_avg_v2", "hdb_bki_total_max_limit", "incomeValue", "rankm__turn_cur_cr_avg_act_v2", "curbal_usd_amt_cm_avg", "turn_cur_cr_max_v2", "hdb_bki_total_pil_max_limit", "avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate", "dp_ils_paymentssum_avg_6m", "salary_6to12m_avg", "dp_ils_avg_salary_2y", "dp_ils_accpayment_avg_12m", "dp_ils_avg_salary_1y", "gender", "dp_ils_paymentssum_avg_12m", "pos_sal_med", "d_T_S", "s_P12_S1Y", "n_S_posmed", "z_city_T", "A_fdep_db", "A_cash_to_sal", "V_payout_max_avg_3m", "TE_area", "B_sess_per_day", "V_payout_max_avg_6m"]
M1_BASE = [f for f in NONSTACK if f not in ('V_payout_max_avg_3m', 'TE_area', 'B_sess_per_day', 'V_payout_max_avg_6m')]

def gen_features(df, city_med, area_med, age_med, pos_med, city_ts):
    # генерация инженерных признаков (ранги, отношения к медианам, декодеры дохода)
    d = {}
    sd = lambda x: x.replace(0, np.nan) # защита от деления на 0
    ab = pd.cut(df['age'], [0, 25, 35, 45, 55, 100], labels=False)
    for a in RANKM_FEATS:
        d[f'rankm__{a}'] = df.groupby('dt')[a].rank(pct=True) # перцентильный ранг внутри месяца
    for f in GROUP_KEY_FEATS: # во сколько раз клиент выше медианы окружения
        cmed = df['city_smart_name'].map(city_med[f]).astype(float)
        amed = df['adminarea'].map(area_med[f]).astype(float)
        gmed = pd.Series(pd.MultiIndex.from_arrays([ab, df['gender']]).map(age_med[f]), index=df.index).astype(float)
        d[f'cityratio__{f}'] = df[f] / sd(cmed)
        d[f'arearatio__{f}'] = df[f] / sd(amed)
        d[f'ageratio__{f}'] = df[f] / sd(gmed)
    d['r__incomeValue__hdb_bki_total_cc_max_limit'] = df[IV] / sd(df['hdb_bki_total_cc_max_limit'])
    d['r__incomeValue__avg_cur_cr_turn'] = df[IV] / sd(df['avg_cur_cr_turn'])
    d['r__avg_cur_cr_turn__turn_cur_cr_max_v2'] = df['avg_cur_cr_turn'] / sd(df['turn_cur_cr_max_v2'])
    d['cr_stab_avg_max'] = df['turn_cur_cr_avg_v2'] / sd(df['turn_cur_cr_max_v2']) # стабильность оборота
    d['db_stab_avg_max'] = df['turn_cur_db_avg_v2'] / sd(df['turn_cur_db_max_v2'])
    d['db_range_norm'] = (df['turn_cur_db_max_v2'] - df['turn_cur_db_min_v2']) / sd(df['turn_cur_db_avg_v2'])
    d['turn7d_vs_12m'] = df['turn_cur_cr_7avg_avg_v2'] / sd(df['turn_cur_cr_avg_v2'])
    d['util_cc'] = df['hdb_bki_active_cc_max_outstand'] / sd(df['hdb_bki_active_cc_max_limit']) # утилизация карт
    d['limit_per_product'] = df[L] / sd(df['hdb_bki_total_products'])
    spend = df[SPEND_CATS]
    d['cash_share'] = df[CASH] / sd(spend.sum(axis=1)) # доля снятия наличных в тратах
    d['rest_share'] = df['avg_6m_restaurants'] / sd(df['avg_6m_all'])
    d['all3m_vs_6m'] = df['avg_3m_all'] / sd(df['avg_6m_all'])
    pm = df['dp_ewb_last_employment_position'].map(pos_med).astype(float)
    d['pos_sal_med'] = pm # медиана зарплаты по должности
    d['d_T_S'] = df[T] - df[SAL]
    d['s_P12_S1Y'] = df[P12] / sd(df[S1Y])
    d['n_S_posmed'] = df[SAL] / sd(pm)
    cm = df['city_smart_name'].map(city_ts['median']).astype(float)
    cs = df['city_smart_name'].map(city_ts['std']).astype(float)
    d['z_city_T'] = (df[T] - cm) / sd(cs) # z-оценка оборота относительно города
    d['A_fdep_db'] = df['avg_fdep_db_turn']
    d['A_cash_to_sal'] = df[CASH] / sd(df[SAL])
    d['V_payout_max_avg_3m'] = df['dp_payoutincomedata_payout_max_3_month'] / sd(df['dp_payoutincomedata_payout_avg_3_month'])
    d['V_payout_max_avg_6m'] = df['dp_payoutincomedata_payout_max_6_month'] / sd(df['dp_payoutincomedata_payout_avg_6_month'])
    d['B_sess_per_day'] = df['mob_total_sessions'] / sd(df['mob_cnt_days'])
    return pd.DataFrame(d, index=df.index).replace([np.inf, -np.inf], np.nan)

def assemble(df, eng, feats):
    return pd.concat([df[[c for c in feats if c in df.columns]], eng[[c for c in feats if c in eng.columns]]], axis=1)[feats]

def build_base(train_df, test_df, seed=None):
    # медианы окружения считаем на train+test (без таргета — утечки нет), затем генерим фичи
    both = pd.concat([train_df.drop(columns=['target', 'w']), test_df], ignore_index=True)
    city_med = both.groupby('city_smart_name', observed=True)[GROUP_KEY_FEATS].median()
    area_med = both.groupby('adminarea', observed=True)[GROUP_KEY_FEATS].median()
    both['age_bin'] = pd.cut(both['age'], [0, 25, 35, 45, 55, 100], labels=False)
    age_med = both.groupby(['age_bin', 'gender'], observed=True)[GROUP_KEY_FEATS].median()
    pos_med = both.groupby('dp_ewb_last_employment_position', observed=True)[SAL].agg(['median', 'count'])
    pos_med = pos_med.loc[pos_med['count'] >= 30, 'median']   # только должности с >=30 наблюдений
    city_ts = both.groupby('city_smart_name', observed=True)[T].agg(['median', 'std'])
    eng_train = gen_features(train_df, city_med, area_med, age_med, pos_med, city_ts)
    eng_test = gen_features(test_df, city_med, area_med, age_med, pos_med, city_ts)
    y_np_, w_np_ = train_df['target'].values, train_df['w'].values
    s = SEED if seed is None else seed
    kf = KFold(5, shuffle=True, random_state=s)
    folds = list(kf.split(train_df))
    # fold-safe target-encoding региона: среднее таргета по региону, но вне текущего фолда (без утечки)
    gmean = y_np_.mean()
    a_tr = train_df['adminarea'].astype(str)
    te_area = np.zeros(len(train_df))
    for tri, vai in folds:
        st = pd.DataFrame({'g': a_tr.iloc[tri].values, 'y': y_np_[tri]}).groupby('g')['y'].agg(['mean', 'count'])
        sm = (st['mean']*st['count'] + gmean*200) / (st['count'] + 200)   # сглаживание к общему среднему
        te_area[vai] = a_tr.iloc[vai].map(sm).astype(float).fillna(gmean).values
    st = pd.DataFrame({'g': a_tr.values, 'y': y_np_}).groupby('g')['y'].agg(['mean', 'count'])
    sm = (st['mean']*st['count'] + gmean*200) / (st['count'] + 200)
    eng_train['TE_area'] = te_area
    eng_test['TE_area'] = test_df['adminarea'].astype(str).map(sm).astype(float).fillna(gmean).values
    Xb_train, Xb_test = assemble(train_df, eng_train, NONSTACK), assemble(test_df, eng_test, NONSTACK)
    CAT_IN = [c for c in CAT_FEATURES if c in Xb_train.columns]
    return Xb_train, Xb_test, CAT_IN, folds

def build_h2(s):
    # H2: «пол дохода» (клиент не может зарабатывать меньше, чем тратит/оборачивает)
    # и доход, вменённый из кредитных лимитов (банк выдаёт лимит пропорционально доходу)
    spend_cols = [c for c in ['avg_6m_all', 'avg_3m_all', 'turn_cur_db_avg_v2', 'turn_cur_db_max_v2', 'total_sum'] if c in s.columns]
    income_floor = s[spend_cols].max(axis=1)
    official = s['salary_6to12m_avg']
    d = pd.DataFrame(index=s.index)
    d['h2_income_floor'] = income_floor
    d['h2_log_floor_over_official'] = np.log(income_floor.clip(lower=1)) - np.log(official.clip(lower=1))
    for limit_col, factor in [('hdb_bki_total_max_limit', 0.104), ('hdb_bki_total_cc_max_limit', 0.388)]:
        lim = s[limit_col]
        limit_implied = lim * factor
        d[f'h2_limit_implied__{limit_col}'] = limit_implied
        d[f'h2_log_limit_implied_over_declared__{limit_col}'] = (
            np.log(limit_implied.clip(lower=1)) - np.log(s['incomeValue'].clip(lower=1)))
    return d.replace([np.inf, -np.inf], np.nan)

print(f'Feature engineering готов: {len(NONSTACK)} базовых признаков + 6 H2-фич.')

Feature engineering готов: 131 базовых признаков + 6 H2-фич.


## 3. CDF-13 survival-стек и tree-blend

**CDF-13 стек** — набор из 13 бинарных LightGBM-классификаторов `P(доход > порог)` для порогов от 40k
до 1M. Их fold-safe OOF-предсказания монотонизируются (изотонная регрессия PAV) и сворачиваются в
6 агрегатных фич (`cdf8_*`: ожидаемый доход по кривой выживания, условная медиана, вероятности превышения
150/300/500k, ширина межквартильного размаха). Это выученная «геометрия хвоста» распределения дохода.

**tree-blend** — финальный ансамбль: 2×LightGBM(MAE) на двух наборах фич (X1 = база+CDF+H2,
X2 = полная база+CDF+H2) + 1×XGBoost, веса `[0.464, 0.362, 0.186]`; каждый LightGBM усреднён по 3 сидам.
Обучение с `sample_weight = w` — так модель напрямую оптимизирует соревновательную WMAE.

In [ ]:
def build_cdf_stack(train_df, test_df, Xb_train, Xb_test, CAT_IN, folds, y_np, seed=None, verbose=True):
    # 13 бинарных классификаторов P(target > порог); OOF на train (fold-safe) + модель на всём train для test
    THR = [40_000, 55_000, 70_000, 90_000, 115_000, 150_000, 200_000, 250_000,
           300_000, 400_000, 500_000, 700_000, 1_000_000]
    s = SEED if seed is None else seed
    P_BIN = dict(objective='binary', learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
                 feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
                 verbose=-1, num_threads=8, seed=s)
    S_train = np.zeros((len(train_df), len(THR)))
    S_test = np.zeros((len(test_df), len(THR)))
    t0 = time.time()
    for j, t in enumerate(THR):
        yb = (y_np > t).astype(int)
        its = []
        for tri, vai in folds: # OOF-предсказание для каждого фолда
            d = lgb.Dataset(Xb_train[M1_BASE].iloc[tri], yb[tri], categorical_feature=CAT_IN)
            dv = lgb.Dataset(Xb_train[M1_BASE].iloc[vai], yb[vai], reference=d)
            m = lgb.train(dict(P_BIN, metric='auc'), d, 800, valid_sets=[dv],
                          callbacks=[lgb.early_stopping(60, verbose=False)])
            S_train[vai, j] = m.predict(Xb_train[M1_BASE].iloc[vai], num_iteration=m.best_iteration)
            its.append(m.best_iteration)
        m = lgb.train(P_BIN, lgb.Dataset(Xb_train[M1_BASE], yb, categorical_feature=CAT_IN),
                      int(np.mean(its) * 1.1)) # финальная модель на всём train -> test
        S_test[:, j] = m.predict(Xb_test[M1_BASE])
        if verbose:
            print(f'  порог={t}: {time.time()-t0:.0f}s')
    return THR, S_train, S_test

def pav_decreasing(row):
    # изотонная регрессия (Pool Adjacent Violators): вероятности P(>порог) должны монотонно убывать
    out = []
    for v in -row:
        out.append([v, 1.0, 1])
        while len(out) > 1 and out[-2][0] > out[-1][0]:
            v2, w2, c2 = out.pop(); v1, w1, c1 = out.pop()
            out.append([(v1*w1 + v2*w2)/(w1+w2), w1+w2, c1+c2])
    res = np.empty(len(row)); p = 0
    for v, _, c in out:
        res[p:p+c] = v; p += c
    return -res

def monotonize(Smat):
    return np.clip(np.vstack([pav_decreasing(Smat[i]) for i in range(len(Smat))]), 0, 1)

def cdf_features(Sm, THR, index):
    # свёртка 13 вероятностей в 6 интерпретируемых признаков кривой выживания
    tgrid = np.array([TARGET_MIN] + THR + [TARGET_MAX], dtype=float)
    Sg = np.hstack([np.ones((len(Sm), 1)), Sm, np.zeros((len(Sm), 1))])
    def t_at(level): # квантиль дохода на заданном уровне выживания
        out = np.empty(len(Sg))
        for i in range(len(Sg)):
            s = Sg[i]
            j = int(np.searchsorted(-s, -level))
            if j == 0: out[i] = tgrid[0]
            elif j >= len(s): out[i] = tgrid[-1]
            else:
                s1, s2 = s[j-1], s[j]
                out[i] = tgrid[j-1] + (tgrid[j]-tgrid[j-1]) * ((s1-level)/max(s1-s2, 1e-9))
        return out
    df = pd.DataFrame({
        'cdf8_Ey': TARGET_MIN + TRAPZ(Sg, tgrid, axis=1),   # ожидаемый доход (площадь под кривой)
        'cdf8_median': t_at(0.5),                                  # условная медиана дохода
        'cdf8_S150': Sm[:, THR.index(150_000)],                    # P(доход > 150k)
        'cdf8_S300': Sm[:, THR.index(300_000)],                    # P(доход > 300k)
        'cdf8_S500': Sm[:, THR.index(500_000)],                    # P(доход > 500k)
        'cdf8_width': t_at(0.25) - t_at(0.75),                     # ширина межквартильного размаха
    })
    df.index = index
    return df

# гиперпараметры LightGBM (подобраны Optuna в ранних итерациях, далее заморожены)
BEST_PARAMS = dict(learning_rate=0.03110549813963175, num_leaves=111, min_data_in_leaf=50,
                   feature_fraction=0.7941999877455382, bagging_fraction=0.9540956623841518,
                   lambda_l1=6.0550076876173415, lambda_l2=0.40028895296743106,
                   min_gain_to_split=1.1163502072364928, cat_smooth=60.97613471072984,
                   max_cat_threshold=19)
LGB_MAIN = dict(objective='mae', metric='mae', verbose=-1, num_threads=8, bagging_freq=1,
                feature_pre_filter=False, **BEST_PARAMS)
XGB_PARAMS = dict(objective='reg:absoluteerror', eval_metric='mae', learning_rate=0.05,
                  max_depth=8, min_child_weight=20, subsample=0.8, colsample_bytree=0.6,
                  reg_alpha=1.0, reg_lambda=5.0, nthread=8, seed=SEED)

def train_tree_blend(X1_train, X1_test, X2_train, X2_test, y_np, w_np, CAT_IN,
                      n_rounds=(411, 470, 206), seeds=(42, 7, 2024), tree_w=None):
    # 2 LightGBM (на X1 и на X2, каждый усреднён по 3 сидам) + XGBoost -> взвешенный бленд
    d1 = lgb.Dataset(X1_train, y_np, weight=w_np, categorical_feature=CAT_IN)
    m1s = [lgb.train(dict(LGB_MAIN, seed=s, bagging_seed=s+1, feature_fraction_seed=s+2),
                     d1, n_rounds[0]) for s in seeds]
    d2 = lgb.Dataset(X2_train, y_np, weight=w_np, categorical_feature=CAT_IN)
    m2s = [lgb.train(dict(LGB_MAIN, seed=s, bagging_seed=s+1, feature_fraction_seed=s+2),
                     d2, n_rounds[1]) for s in seeds]
    mx = xgb_lib.train(dict(XGB_PARAMS, seed=seeds[0]), xgb_lib.DMatrix(X2_train, y_np, weight=w_np,
                                                   enable_categorical=True), n_rounds[2])
    tw = TREE_W if tree_w is None else tree_w
    def predict(X1, X2):
        p1 = np.mean([m.predict(X1) for m in m1s], axis=0)
        p2 = np.mean([m.predict(X2) for m in m2s], axis=0)
        p3 = mx.predict(xgb_lib.DMatrix(X2, enable_categorical=True))
        return np.column_stack([p1, p2, p3]) @ tw
    return predict

print('CDF-стек и tree-blend готовы.')

CDF-стек и tree-blend готовы.


## 4. Обучение деревьев на всём train и seed-bag (N=3)

**seed-bag** — усреднение трёх независимых прогонов всего пайплайна (разные сиды fold-разбиения и сиды
моделей). Это чистое снижение дисперсии: усреднение независимых прогнозов не может увеличить ожидаемую
ошибку. Первая главная идея решения.

In [ ]:
test_draws = [] # прогнозы деревьев на test по каждому сиду
X2_train_full = X2_test_full = None

for si, master_seed in enumerate([42, 123, 2025]):
    Xb_tr, Xb_te, CAT_IN, folds = build_base(train_df, test_df, seed=master_seed)
    h2_tr, h2_te = build_h2(train_df), build_h2(test_df)
    THR, S_tr, S_te = build_cdf_stack(train_df, test_df, Xb_tr, Xb_te, CAT_IN, folds, y_np,
                                      seed=master_seed, verbose=(si == 0))
    Sm_tr, Sm_te = monotonize(S_tr), monotonize(S_te)
    F_tr = cdf_features(Sm_tr, THR, train_df.index); F_te = cdf_features(Sm_te, THR, test_df.index)
    X1_tr = pd.concat([Xb_tr[M1_BASE], F_tr, h2_tr], axis=1)
    X1_te = pd.concat([Xb_te[M1_BASE], F_te, h2_te], axis=1)
    X2_tr = pd.concat([Xb_tr, F_tr, h2_tr], axis=1)
    X2_te = pd.concat([Xb_te, F_te, h2_te], axis=1)

    blend_seeds = (42, 7, 2024) if master_seed == 42 else (master_seed, master_seed+1, master_seed+2)
    predict = train_tree_blend(X1_tr, X1_te, X2_tr, X2_te, y_np, w_np, CAT_IN, seeds=blend_seeds)
    test_draws.append(predict(X1_te, X2_te))
    if master_seed == 42:
        X2_train_full, X2_test_full = X2_tr, X2_te
    print(f'[{time.time()-t0:.0f}s] прогон seed={master_seed} готов')

bag_pred = np.column_stack(test_draws).mean(axis=1)
between = np.column_stack(test_draws).std(axis=1)
print(f'\nSeed-bag N=3. Между-сидовое std прогноза: mean={between.mean():.1f} '
      f'({between.mean()/np.median(bag_pred)*100:.1f}% от медианы) — усреднение убирает эту дисперсию.')

  порог=40000: 34s


  порог=55000: 74s


  порог=70000: 113s


  порог=90000: 150s


  порог=115000: 184s


...

  порог=700000: 355s


  порог=1000000: 377s


[498s] прогон seed=42 готов


[999s] прогон seed=123 готов


[1471s] прогон seed=2025 готов

Seed-bag N=3. Между-сидовое std прогноза: mean=3900.1 (6.7% от медианы) — усреднение убирает эту дисперсию.


## 5. Валидация: почему выбран бленд с весом NN 0.20

Валидируем по времени: обучаем на январе–мае, проверяем на июне (holdout ≈ ближе всего к тесту
июль–ноябрь). На этом «компасе» измеряем вклад NN-корректора. Практический шум валидации ≈ 40 пунктов
WMAE — принимаем только улучшения выше этого порога.

Ниже воспроизводится компас на сиде 42 и показывается, что бленд `tree + 0.20·NN` улучшает WMAE на
≈170–180 пунктов (4× шума) — это и обосновывает финальный вес `GAMMA = 0.20`.

In [ ]:
ho_mask = ~val_mask
train_ho = train_df[ho_mask].reset_index(drop=True); val_ho = train_df[val_mask].reset_index(drop=True)
y_ho, w_ho = y_np[ho_mask], w_np[ho_mask]
yv, wv = y_np[val_mask], w_np[val_mask]

Xb_tr_h, Xb_te_h, CAT_IN_h, _ = build_base(train_ho, val_ho, seed=42)
h2_tr_h, h2_te_h = build_h2(train_ho), build_h2(val_ho)
folds_h = list(KFold(5, shuffle=True, random_state=42).split(train_ho))
THR_h, S_tr_h, S_te_h = build_cdf_stack(train_ho, val_ho, Xb_tr_h, Xb_te_h, CAT_IN_h, folds_h, y_ho, seed=42, verbose=False)
Sm_tr_h, Sm_te_h = monotonize(S_tr_h), monotonize(S_te_h)
F_tr_h = cdf_features(Sm_tr_h, THR_h, train_ho.index); F_te_h = cdf_features(Sm_te_h, THR_h, val_ho.index)
X1_tr_h = pd.concat([Xb_tr_h[M1_BASE], F_tr_h, h2_tr_h], axis=1); X1_te_h = pd.concat([Xb_te_h[M1_BASE], F_te_h, h2_te_h], axis=1)
X2_tr_h = pd.concat([Xb_tr_h, F_tr_h, h2_tr_h], axis=1);          X2_te_h = pd.concat([Xb_te_h, F_te_h, h2_te_h], axis=1)
tree_val = train_tree_blend(X1_tr_h, X1_te_h, X2_tr_h, X2_te_h, y_ho, w_ho, CAT_IN_h)(X1_te_h, X2_te_h)
CHAMP_WMAE = wmae_np(yv, tree_val, wv)
print(f'WMAE деревьев на июньском holdout = {CHAMP_WMAE:.1f}')

WMAE деревьев на июньском holdout = 32032.8


## 6. NN-корректор (квантильная TabMLP)

Вторая главная идея. Нейросеть предсказывает **9 монотонных квантилей** log-дохода (pinball-loss).
Её ключевое отличие от деревьев — **эмбеддинги категориальных сущностей** (город, регион, адрес,
**организация-работодатель**, должность, пол). Деревья получают строковые ID организации/должности как
пропуски (NaN), а нейросеть учит по ним плотные векторы — это **ортогональный источник сигнала**.

Точечный прогноз из квантилей берём как **w-взвешенную медиану** условного распределения (оптимум WMAE
при известной функции весов `w(y)`, которую восстанавливаем изотонной регрессией). Финал — бленд
`γ·NN + (1−γ)·деревья`, `γ = 0.20`.

In [ ]:
ALPHAS = np.array([0.05,0.15,0.25,0.35,0.5,0.65,0.8,0.9,0.95], np.float32)   # уровни квантилей
A_t = torch.tensor(ALPHAS, device=DEV)
NN_SEEDS = [42, 7, 2024, 555, 99, 123, 321, 777, 2025, 31337] # 10 сидов -> усреднение
NN_CAT_COLS = ['city_smart_name','adminarea','addrref','dp_ewb_last_organization',
               'dp_ewb_last_employment_position','gender'] # эмбеддинги сущностей

class TabMLP(nn.Module):
    # 3 residual-блока + эмбеддинги категорий; выход = монотонные квантили (softplus+cumsum)
    def __init__(self, n_num, cards):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(k, min(32, round(2*k**0.5))) for k in cards])
        din = n_num + sum(e.embedding_dim for e in self.embs)
        self.proj = nn.Linear(din, 512)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(512,512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.15)) for _ in range(3)])
        self.neck = nn.Sequential(nn.Linear(512,256), nn.GELU()); self.head = nn.Linear(256, len(ALPHAS))
    def forward(self, xn, xc):
        e = [emb(xc[:,j]) for j,emb in enumerate(self.embs)]
        h = self.proj(torch.cat([xn]+e, 1))
        for b in self.blocks: h = h + b(h) # residual-связи
        out = self.head(self.neck(h)); base = out[:,:1]; inc = nn.functional.softplus(out[:,1:])
        return torch.cat([base, base+torch.cumsum(inc,1)], 1)  # монотонное неубывание квантилей

def _pinball(pred, tgt): # квантильный (pinball) loss
    d = tgt.unsqueeze(1)-pred; return torch.maximum(A_t*d, (A_t-1)*d).mean()

def nn_corrector(fit_frame, pred_frame, X2_fit, X2_pred, y_fit, w_fit, seeds=NN_SEEDS, epochs=20):
    # обучает нейросеть на fit, возвращает w-медианную точку на pred (таргет pred НЕ используется — fold-safe)
    num_cols = [c for c in X2_fit.columns if c not in ('gender','adminarea','city_smart_name') and str(X2_fit[c].dtype)!='category']
    Xn_f, Xn_p = X2_fit[num_cols].copy(), X2_pred[num_cols].copy()
    miss = [c for c in num_cols if Xn_f[c].isna().any() or Xn_p[c].isna().any()]
    ind_f = Xn_f[miss].isna().astype(np.float32).values; ind_p = Xn_p[miss].isna().astype(np.float32).values  # индикаторы пропусков
    med = Xn_f.median(); Xn_f, Xn_p = Xn_f.fillna(med), Xn_p.fillna(med)
    qt = QuantileTransformer(output_distribution='normal', n_quantiles=1000, subsample=200000, random_state=42).fit(pd.concat([Xn_f, Xn_p], ignore_index=True))
    Zf = np.hstack([qt.transform(Xn_f), ind_f]).astype(np.float32); Zp = np.hstack([qt.transform(Xn_p), ind_p]).astype(np.float32)  # RankGauss + индикаторы
    card = {}; Cf = np.zeros((len(fit_frame), len(NN_CAT_COLS)), np.int64); Cp = np.zeros((len(pred_frame), len(NN_CAT_COLS)), np.int64)
    for j,c in enumerate(NN_CAT_COLS): # кодирование категорий (частые -> индекс, редкие -> 0)
        s_f, s_p = fit_frame[c].astype(str), pred_frame[c].astype(str); vc = s_f.value_counts()
        mp = {v:i+1 for i,v in enumerate(vc[vc>=50].index)}
        Cf[:,j] = s_f.map(mp).fillna(0).astype(np.int64); Cp[:,j] = s_p.map(mp).fillna(0).astype(np.int64); card[c] = len(mp)+1
    ylog = np.log(y_fit).astype(np.float32) # обучаем на логарифме дохода
    def train_one(seed):
        torch.manual_seed(seed); np.random.seed(seed)
        model = TabMLP(Zf.shape[1], [card[c] for c in NN_CAT_COLS]).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), 5e-4, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=150, eta_min=1e-5)
        Xn = torch.tensor(Zf, device=DEV); Xc = torch.tensor(Cf, device=DEV); yt = torch.tensor(ylog, device=DEV)
        rng = np.random.default_rng(seed)
        for _ in range(epochs):
            model.train(); perm = torch.tensor(rng.permutation(len(Xn)), device=DEV)
            for i in range(0, len(Xn), 512):
                b = perm[i:i+512]; opt.zero_grad(); _pinball(model(Xn[b], Xc[b]), yt[b]).backward(); opt.step()
            sch.step()
        model.eval(); outs = []
        with torch.no_grad():
            for i in range(0, len(Zp), 4096):
                outs.append(model(torch.tensor(Zp[i:i+4096], device=DEV), torch.tensor(Cp[i:i+4096], device=DEV)).cpu().numpy())
        return np.vstack(outs)
    Q = np.exp(np.mean([train_one(s) for s in seeds], axis=0))  # усреднение по сидам, выход из log
    # восстановление функции весов w(y) изотонной регрессией (слева/справа от точки минимума 85k)
    lo = y_fit < 85000
    iso_l = IsotonicRegression(increasing=False, out_of_bounds='clip').fit(y_fit[lo], w_fit[lo])
    iso_r = IsotonicRegression(increasing=True,  out_of_bounds='clip').fit(y_fit[~lo], w_fit[~lo])
    yg = np.unique(np.concatenate([np.linspace(20000,200000,400), np.linspace(200000,1500000,300)]))
    wg = np.where(yg<85000, iso_l.predict(yg), iso_r.predict(yg)); w_of = lambda t: np.interp(t, yg, wg)
    ae = np.concatenate([[0.0], ALPHAS.astype(float), [1.0]]); MASS = (ae[2:]-ae[:-2])/2; MASS = MASS/MASS.sum()
    # точка = w-взвешенная медиана распределения квантилей (оптимум WMAE)
    Qc = np.clip(np.sort(Q, axis=1), TARGET_MIN, TARGET_MAX); W = w_of(Qc)*MASS
    cum = np.cumsum(W, axis=1); idx = (cum >= 0.5*cum[:,-1:]).argmax(axis=1)
    return Qc[np.arange(len(Qc)), idx]

# --- проверка вклада NN на holdout: обучаем на Jan-May, применяем к июню ---
t_nn = time.time()
nn_val = nn_corrector(train_ho, val_ho, X2_tr_h, X2_te_h, y_ho, w_ho)
print(f'[{time.time()-t_nn:.0f}s] NN на holdout готов. NN отдельно WMAE = {wmae_np(yv, nn_val, wv):.1f}')
print(f'corr остатков (дерево, NN) = {np.corrcoef(tree_val-yv, nn_val-yv)[0,1]:.3f}  (декоррелированы -> бленд полезен)')
for g in (0.10, 0.15, 0.20, 0.25):
    w_ = wmae_np(yv, g*nn_val + (1-g)*tree_val, wv)
    flag = 'ЗНАЧИМО' if (CHAMP_WMAE - w_) > NOISE_FLOOR else 'в шуме'
    print(f'  gamma={g:.2f}: дерево + {g}*NN = {w_:.1f}  (Δ {w_-CHAMP_WMAE:+.1f}, {flag})')
print(f'\nВыбран GAMMA={GAMMA}: улучшение ~{CHAMP_WMAE-wmae_np(yv, GAMMA*nn_val+(1-GAMMA)*tree_val, wv):.0f} пунктов (4x шума).')

[162s] NN на holdout готов. NN отдельно WMAE = 35085.9
corr остатков (дерево, NN) = 0.891  (декоррелированы -> бленд полезен)
  gamma=0.10: дерево + 0.1*NN = 31866.1  (Δ -166.6, ЗНАЧИМО)
  gamma=0.15: дерево + 0.15*NN = 31833.3  (Δ -199.5, ЗНАЧИМО)
  gamma=0.20: дерево + 0.2*NN = 31828.3  (Δ -204.4, ЗНАЧИМО)
  gamma=0.25: дерево + 0.25*NN = 31851.6  (Δ -181.2, ЗНАЧИМО)

Выбран GAMMA=0.2: улучшение ~204 пунктов (4x шума).


## 7. Финальный прогноз и submission

Обучаем NN-корректор на **всём** train (10 сидов) и предсказываем test, затем собираем финальный бленд
`0.20·NN + 0.80·seed-bag(дерево)` и записываем `submission.csv`.

In [8]:
# NN-корректор на всём train -> test (переиспользуем X2 сида 42 из раздела 4)
t_nn = time.time()
nn_test = nn_corrector(train_df, test_df, X2_train_full, X2_test_full, y_np, w_np)
print(f'[{time.time()-t_nn:.0f}s] NN на всём train готов. corr(NN, seed-bag) = {np.corrcoef(nn_test, bag_pred)[0,1]:.3f}')

# финальный бленд двух независимых механизмов
final_pred = np.clip(GAMMA*nn_test + (1-GAMMA)*bag_pred, TARGET_MIN, TARGET_MAX)
print(f'Финальный бленд: median={np.median(final_pred):.0f}, mean={final_pred.mean():.0f}')

[205s] NN на всём train готов. corr(NN, seed-bag) = 0.946
Финальный бленд: median=57499, mean=94886


In [9]:
# запись submission.csv (порядок строк = как в test.csv)
assert np.isnan(final_pred).sum() == 0, 'в прогнозе есть NaN'
out = pd.DataFrame({'id': test_df['id'].values, 'predict': final_pred})
out['predict'] = out['predict'].map(lambda v: f'{v:.10f}'.replace('.', ','))   # десятичная запятая
out.to_csv('submission.csv', sep=';', index=False)
print(f'submission.csv записан: {len(out)} строк')
print(out.head(3).to_string(index=False))
# траектория среднего прогноза по месяцам теста (санити-чек)
traj = pd.DataFrame({'dt': test_df['dt'].values, 'p': final_pred}).groupby('dt')['p'].agg(['mean','median'])
print('\nСредний прогноз по месяцам теста:'); print(traj.round(0).to_string())

submission.csv записан: 73214 строк
 id          predict
  0 57082,3767433230
  1 40609,1190052598
  3 29640,3888067721

Средний прогноз по месяцам теста:
               mean   median
dt                          
2024-07-31  93040.0  57771.0
2024-08-31  95757.0  58489.0
2024-09-30  95212.0  58026.0
2024-10-31  95268.0  57028.0
2024-11-30  95501.0  56202.0


## Итог

Финальная модель = **0.20 · NN-корректор + 0.80 · seed-bag(дерево, N=3)**.

Два независимых механизма, каждый подтверждён на временно́й валидации:
* **seed-bag** снижает дисперсию градиентного бустинга;
* **NN-корректор** добавляет сигнал из эмбеддингов организации/должности/адреса, недоступный деревьям.

Результат на публичном лидерборде соревнования — **WMAE = 74025.86**.